# 05 — Evaluation

Compare the trained GraphSAGE model against the Douglas-Peucker baseline
on the **test set** (Wageningen).

Metrics:
- **F1 score** — precision/recall balance for survival prediction
- **IoU** — spatial overlap between predicted and true BRT coverage
- **Hausdorff distance** — worst-case boundary deviation
- **Polygon count ratio** — how well predicted polygon count matches BRT

Run **03_baseline.ipynb** and **04_model.ipynb** first.


## 0. CONFIG

In [ ]:
from pathlib import Path

CONFIG = {
    "output_root":         Path("processed"),
    "model_dir":           Path("models"),
    "crs":                 "EPSG:28992",

    # Must match notebook 04
    "bgt_class_col":       "plus-fysiekVoorkomenWegdeel",
    "adjacency_distance_m": 0.5,
    "node_features": [
        "area", "perimeter", "compactness",
        "aspect_ratio", "n_vertices", "class_encoded",
    ],
    "hidden_dim":   64,
    "n_layers":     3,
    "dropout":      0.3,

    # Evaluate on this many test tiles (None = all)
    "eval_n_tiles": None,
}

## 1. Imports

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn   import SAGEConv

from shapely.errors import ShapelyDeprecationWarning
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2. Load encoder and model

In [ ]:
def load_encoder(path):
    with open(path) as f:
        data = json.load(f)
    class Enc: pass
    enc = Enc()
    enc.unknown_label = data["unknown_label"]
    enc.classes_      = data["classes"]
    enc.label_to_int  = data["label_to_int"]
    enc.int_to_label  = {int(k): v for k, v in data["int_to_label"].items()}
    return enc

bgt_encoder = load_encoder(CONFIG["output_root"] / "bgt_encoder.json")
N_CLASSES   = len(bgt_encoder.classes_)
N_FEATURES  = len(CONFIG["node_features"])

# Re-define model (self-contained)
class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.convs   = nn.ModuleList()
        self.norms   = nn.ModuleList()
        self.dropout = dropout
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        self.norms.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(n_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.norms.append(nn.BatchNorm1d(hidden_dim))
        self.head = nn.Linear(hidden_dim, 2)
    def forward(self, x, edge_index):
        for conv, norm in zip(self.convs, self.norms):
            x = conv(x, edge_index)
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(x)

model = GraphSAGE(N_FEATURES, CONFIG["hidden_dim"], CONFIG["n_layers"], CONFIG["dropout"]).to(DEVICE)
model.load_state_dict(torch.load(CONFIG["model_dir"] / "best_graphsage.pt", map_location=DEVICE))
model.eval()
print(f"Model loaded — {sum(p.numel() for p in model.parameters()):,} parameters")

## 3. Graph construction helpers (re-defined for self-containment)

In [ ]:
def compute_node_features(gdf, feature_names, class_col_encoded):
    df = pd.DataFrame(index=gdf.index)
    area      = gdf.geometry.area
    perimeter = gdf.geometry.length
    df["area"]        = np.log1p(area)
    df["perimeter"]   = np.log1p(perimeter)
    df["compactness"] = (4 * np.pi * area) / (perimeter ** 2 + 1e-9)
    bounds = gdf.geometry.bounds
    w = (bounds["maxx"] - bounds["minx"]).clip(lower=1e-6)
    h = (bounds["maxy"] - bounds["miny"]).clip(lower=1e-6)
    df["aspect_ratio"] = np.log1p(w / h)
    df["n_vertices"] = np.log1p(gdf.geometry.apply(
        lambda g: len(g.exterior.coords) if g.geom_type == "Polygon"
        else sum(len(p.exterior.coords) for p in g.geoms)
    ))
    df["class_encoded"] = gdf[class_col_encoded] / max(N_CLASSES - 1, 1) \
        if class_col_encoded in gdf.columns else 0.0
    matrix = df[feature_names].fillna(0.0).values.astype(np.float32)
    mean = matrix.mean(axis=0); std = matrix.std(axis=0) + 1e-8
    return (matrix - mean) / std

def build_adjacency(gdf, dist):
    if len(gdf) == 0:
        return torch.zeros((2, 0), dtype=torch.long)
    buffered = gdf.copy()
    buffered["geometry"] = gdf.geometry.buffer(dist)
    joined = gpd.sjoin(
        buffered[["geometry"]].reset_index().rename(columns={"index": "src"}),
        gdf[["geometry"]].reset_index().rename(columns={"index": "dst"}),
        how="inner", predicate="intersects"
    )
    edges = joined[["src", "dst"]].values
    edges = edges[edges[:, 0] != edges[:, 1]]
    edges = np.unique(np.sort(edges, axis=1), axis=0)
    edges_bi = np.concatenate([edges, edges[:, ::-1]], axis=0)
    return torch.tensor(edges_bi.T, dtype=torch.long)

def tile_to_graph(bgt, config):
    bgt = bgt.reset_index(drop=True)
    enc_col = f"{config['bgt_class_col']}_encoded"
    x = compute_node_features(bgt, config["node_features"], enc_col)
    x = torch.tensor(x, dtype=torch.float)
    edge_index = build_adjacency(bgt, config["adjacency_distance_m"])
    y = torch.tensor(bgt["survived"].fillna(0).values, dtype=torch.long) \
        if "survived" in bgt.columns else torch.zeros(len(bgt), dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)

print("Graph helpers defined.")

## 4. Evaluate GNN on test set

In [ ]:
with open(CONFIG["output_root"] / "tile_index.json") as f:
    tile_index = json.load(f)

test_tiles = tile_index["test"]
if CONFIG["eval_n_tiles"]:
    test_tiles = test_tiles[:CONFIG["eval_n_tiles"]]

print(f"Evaluating GNN on {len(test_tiles)} test tiles (Wageningen)...")

all_preds, all_labels = [], []
gnn_results = []

def iou_union(pred_gdf, target_gdf):
    try:
        pu = pred_gdf.geometry.union_all()
        tu = target_gdf.geometry.union_all()
        inter = pu.intersection(tu).area
        union = pu.union(tu).area
        return inter / union if union > 0 else 0.0
    except Exception:
        return 0.0

def evaluate_spatial(pred_gdf, target_gdf):
    if len(pred_gdf) == 0 or len(target_gdf) == 0:
        return {"iou": 0.0, "hausdorff": float("nan"), "count_ratio": 0.0}
    pu = pred_gdf.geometry.union_all()
    tu = target_gdf.geometry.union_all()
    iou_val = iou_union(pred_gdf, target_gdf)
    try:
        hausdorff = pu.hausdorff_distance(tu)
    except Exception:
        hausdorff = float("nan")
    return {
        "iou":         round(iou_val, 4),
        "hausdorff":   round(hausdorff, 2),
        "count_ratio": round(len(pred_gdf) / max(len(target_gdf), 1), 3),
    }

for i, tile in enumerate(test_tiles):
    bgt = gpd.read_file(tile["bgt"])
    brt = gpd.read_file(tile["brt"])

    if len(bgt) < 2:
        continue

    graph = tile_to_graph(bgt, CONFIG).to(DEVICE)
    with torch.no_grad():
        logits = model(graph.x, graph.edge_index)
        preds  = logits.argmax(dim=1).cpu().numpy()

    # Collect node-level predictions for classification metrics
    if "survived" in bgt.columns:
        all_preds.append(preds)
        all_labels.append(bgt["survived"].fillna(0).values)

    # Spatial metrics: compare predicted-survived polygons vs BRT
    pred_gdf = bgt[preds == 1]
    spatial  = evaluate_spatial(pred_gdf, brt)
    spatial["tile"] = tile["bgt"]
    spatial["city"] = tile["city"]
    gnn_results.append(spatial)

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(test_tiles)}")

gnn_df = pd.DataFrame(gnn_results)

# Node-level classification metrics
all_preds  = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)
tp = ((all_preds == 1) & (all_labels == 1)).sum()
fp = ((all_preds == 1) & (all_labels == 0)).sum()
fn = ((all_preds == 0) & (all_labels == 1)).sum()
precision = tp / max(tp + fp, 1)
recall    = tp / max(tp + fn, 1)
f1        = 2 * precision * recall / max(precision + recall, 1e-8)
accuracy  = (all_preds == all_labels).mean()

print(f"\nGNN node-level metrics:")
print(f"  Accuracy  : {accuracy*100:.2f}%")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1        : {f1:.4f}")

gnn_df.to_csv(CONFIG["output_root"] / "gnn_results.csv", index=False)
print("\nResults saved.")

## 5. GNN vs Baseline comparison

In [ ]:
baseline_df = pd.read_csv(CONFIG["output_root"] / "baseline_results.csv")

metrics = [
    ("iou",         "IoU",           "↑ better",  True),
    ("hausdorff",   "Hausdorff (m)", "↓ better",  False),
    ("count_ratio", "Count ratio",   "1.0 ideal", None),
]

print("═" * 60)
print("GNN vs BASELINE — Test set (Wageningen)")
print("═" * 60)
print(f"{'Metric':<20} {'Baseline':>12} {'GNN':>12} {'Better?':>10}")
print("-" * 60)

for col, label, note, higher_better in metrics:
    bv = baseline_df[col].mean() if col in baseline_df.columns else float("nan")
    gv = gnn_df[col].mean()      if col in gnn_df.columns      else float("nan")
    if higher_better is True:
        flag = "✓" if gv > bv else "✗"
    elif higher_better is False:
        flag = "✓" if gv < bv else "✗"
    else:
        flag = "✓" if abs(gv - 1) < abs(bv - 1) else "✗"
    print(f"{label:<20} {bv:>12.4f} {gv:>12.4f} {flag:>10}  ({note})")

print("-" * 60)
print(f"{'F1 (survival)':<20} {'—':>12} {f1:>12.4f}")
print(f"{'Precision':<20} {'—':>12} {precision:>12.4f}")
print(f"{'Recall':<20} {'—':>12} {recall:>12.4f}")

## 6. Metric distribution plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (col, label, note, _) in zip(axes, metrics):
    bv = baseline_df[col].dropna() if col in baseline_df.columns else pd.Series(dtype=float)
    gv = gnn_df[col].dropna()      if col in gnn_df.columns      else pd.Series(dtype=float)
    all_vals = pd.concat([bv, gv])
    if len(all_vals) == 0:
        continue
    bins = np.linspace(all_vals.min(), all_vals.max(), 25)
    ax.hist(bv, bins=bins, alpha=0.6, color="steelblue", label="Baseline", density=True)
    ax.hist(gv, bins=bins, alpha=0.6, color="tomato",    label="GNN",      density=True)
    ax.axvline(bv.mean(), color="steelblue", linestyle="--", linewidth=1.5)
    ax.axvline(gv.mean(), color="tomato",    linestyle="--", linewidth=1.5)
    if col == "count_ratio":
        ax.axvline(1.0, color="black", linestyle=":", linewidth=1)
    ax.set_title(f"{label}\n({note})")
    ax.legend(fontsize=8)

plt.suptitle("GNN vs Baseline — Test set metric distributions", fontsize=13)
plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "gnn_vs_baseline.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Visual comparison on a test tile

In [ ]:
def generalize_baseline(bgt, min_area=25.0, dp_tol=1.0):
    out = bgt[bgt.geometry.area >= min_area].copy()
    out["geometry"] = out.geometry.simplify(dp_tol, preserve_topology=True)
    return out[out.geometry.is_valid & ~out.geometry.is_empty]

# Pick tile closest to median GNN IoU
med_idx  = (gnn_df["iou"] - gnn_df["iou"].median()).abs().idxmin()
example  = test_tiles[med_idx]
bgt_ex   = gpd.read_file(example["bgt"])
brt_ex   = gpd.read_file(example["brt"])

# GNN prediction
graph_ex = tile_to_graph(bgt_ex, CONFIG).to(DEVICE)
with torch.no_grad():
    preds_ex = model(graph_ex.x, graph_ex.edge_index).argmax(dim=1).cpu().numpy()
gnn_pred = bgt_ex[preds_ex == 1]

# Baseline prediction
base_pred = generalize_baseline(bgt_ex)

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
plots = [
    (bgt_ex,   "steelblue",    f"BGT input ({len(bgt_ex)} polygons)"),
    (base_pred,"darkorange",   f"Baseline ({len(base_pred)} polygons)"),
    (gnn_pred, "mediumpurple", f"GNN survived ({len(gnn_pred)} polygons)"),
    (brt_ex,   "tomato",       f"BRT target ({len(brt_ex)} polygons)"),
]
for ax, (gdf, color, title) in zip(axes, plots):
    if len(gdf) > 0:
        gdf.plot(ax=ax, color=color, edgecolor="white", linewidth=0.3, alpha=0.85)
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()

gnn_iou  = gnn_df.loc[med_idx, "iou"]
base_iou = baseline_df.iloc[med_idx]["iou"] if med_idx < len(baseline_df) else float("nan")
plt.suptitle(f"Test tile — GNN IoU: {gnn_iou:.3f}  |  Baseline IoU: {base_iou:.3f}", fontsize=12)
plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "test_tile_comparison.png", dpi=150, bbox_inches="tight")
plt.show()